# Part 4 — Algorithm Visualization
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

All charts use **seaborn** + **matplotlib** with a consistent professional style.
Run Cell 1 first (setup), then run any chart cell independently.

| Cell | Chart | What it shows |
|------|-------|---------------|
| **1** | Setup | All algorithms + seaborn/matplotlib style config |
| **2** | Chart 1 — Benchmark Bars | Time / Comparisons / Swaps side by side |
| **3** | Chart 2 — Scalability | Time vs n line chart with direct labels |
| **4** | Chart 3 — Heatmap | Best / Average / Worst case (seaborn heatmap) |
| **5** | Chart 4 — MST Graph | NetworkX spring layout, styled edges |
| **6** | Chart 5 — Recursion Growth | O(n) vs O(2^n) with danger zone shading |
| **7** | Chart 6 — Hanoi Growth | Gradient bar + log-scale line |
| **8** | Chart 7 — Recursion Tree | Factorial chain + Fibonacci binary tree *(bonus)* |
| **9** | Chart 8 — Metrics Breakdown | Full time/comps/swaps vertical bars |
| **10** | Chart 9 — MST Cost | Kruskal vs Prim cost on 10 random graphs |

In [ ]:
# =========================================================
#  CELL 1 — SETUP
#  All algorithms + seaborn/matplotlib style configuration.
#  Run this first; all other cells depend on it.
# =========================================================
import time, random, heapq, math, warnings
from collections import defaultdict
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ── Palette & style ────────────────────────────────────────────────────
PALETTE   = sns.color_palette("deep", 10)
GREEN     = "#10B981"
RED       = "#EF4444"
BLUE      = "#2563EB"
SLATE     = "#64748B"
FIG_BG    = "#F8FAFC"

PAL_NAMES = {n: PALETTE[i] for i, n in enumerate([
    "Bubble Sort","Selection Sort","Insertion Sort","Merge Sort",
    "Quick Sort","Random-Quick Sort","Counting Sort","Radix Sort",
    "Kruskal","Prim"
])}

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor":  FIG_BG,
    "axes.facecolor":    "white",
    "axes.edgecolor":    "#E2E8F0",
    "axes.linewidth":    0.8,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlepad":     12,
    "axes.labelsize":    10,
    "axes.labelcolor":   "#374151",
    "axes.labelpad":     6,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "xtick.color":       "#6B7280",
    "ytick.color":       "#6B7280",
    "grid.color":        "#E5E7EB",
    "grid.linewidth":    0.7,
    "legend.frameon":    True,
    "legend.framealpha": 0.92,
    "legend.fontsize":   9,
    "legend.edgecolor":  "#E2E8F0",
    "figure.dpi":        130,
    "savefig.dpi":       150,
    "savefig.bbox":      "tight",
    "font.family":       "sans-serif",
})

def style_ax(ax, spines=("top","right")):
    for sp in spines: ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)

def label_bars_h(ax, bars, fmt=".2f"):
    xl = ax.get_xlim()[1]
    for bar in bars:
        w = bar.get_width()
        lb = f"{w:{fmt}}" if "f" in fmt else f"{int(w):,}"
        ax.text(w + xl*0.012, bar.get_y() + bar.get_height()/2,
                lb, va="center", ha="left", fontsize=8, color="#374151", fontweight="600")

def label_bars_v(ax, bars, fmt=",d"):
    yl = ax.get_ylim()[1]
    for bar in bars:
        h = bar.get_height()
        if h == 0: continue
        lb = f"{int(h):,}" if fmt == ",d" else f"{h:{fmt}}"
        ax.text(bar.get_x() + bar.get_width()/2, h + yl*0.012,
                lb, ha="center", va="bottom", fontsize=7.5, color="#374151", fontweight="600")

# ── Sorting algorithms ─────────────────────────────────────────────────
def bubble_sort(arr):
    a=arr[:]; n=len(a); c=s=0
    for i in range(n):
        sw=False
        for j in range(n-i-1):
            c+=1
            if a[j]>a[j+1]: a[j],a[j+1]=a[j+1],a[j]; s+=1; sw=True
        if not sw: break
    return a,c,s

def selection_sort(arr):
    a=arr[:]; n=len(a); c=s=0
    for i in range(n):
        mi=i
        for j in range(i+1,n):
            c+=1
            if a[j]<a[mi]: mi=j
        if mi!=i: a[i],a[mi]=a[mi],a[i]; s+=1
    return a,c,s

def insertion_sort(arr):
    a=arr[:]; c=s=0
    for i in range(1,len(a)):
        key=a[i]; j=i-1
        while j>=0:
            c+=1
            if a[j]>key: a[j+1]=a[j]; s+=1; j-=1
            else: break
        a[j+1]=key
    return a,c,s

def merge_sort(arr):
    comps=[0]
    def mg(L,R):
        o=[]; i=j=0
        while i<len(L) and j<len(R):
            comps[0]+=1
            if L[i]<=R[j]: o.append(L[i]); i+=1
            else: o.append(R[j]); j+=1
        o.extend(L[i:]); o.extend(R[j:]); return o
    def s(a):
        if len(a)<=1: return a
        m=len(a)//2; return mg(s(a[:m]),s(a[m:]))
    return s(arr[:]),comps[0],0

def quick_sort(arr):
    c=[0]; s=[0]
    def p(a,lo,hi):
        pv=a[hi]; i=lo-1
        for j in range(lo,hi):
            c[0]+=1
            if a[j]<=pv: i+=1; a[i],a[j]=a[j],a[i]; s[0]+=1
        a[i+1],a[hi]=a[hi],a[i+1]; s[0]+=1; return i+1
    def q(a,lo,hi):
        if lo<hi: pp=p(a,lo,hi); q(a,lo,pp-1); q(a,pp+1,hi)
    a=arr[:]; q(a,0,len(a)-1); return a,c[0],s[0]

def random_quick_sort(arr):
    c=[0]; s=[0]
    def p(a,lo,hi):
        r=random.randint(lo,hi); a[r],a[hi]=a[hi],a[r]; s[0]+=1
        pv=a[hi]; i=lo-1
        for j in range(lo,hi):
            c[0]+=1
            if a[j]<=pv: i+=1; a[i],a[j]=a[j],a[i]; s[0]+=1
        a[i+1],a[hi]=a[hi],a[i+1]; s[0]+=1; return i+1
    def q(a,lo,hi):
        if lo<hi: pp=p(a,lo,hi); q(a,lo,pp-1); q(a,pp+1,hi)
    a=arr[:]; q(a,0,len(a)-1); return a,c[0],s[0]

def counting_sort(arr):
    if not arr: return [],0,0
    a=arr[:]; mx=max(a); cnt=[0]*(mx+1); c=0
    for v in a: cnt[v]+=1; c+=1
    return [i for i,cc in enumerate(cnt) for _ in range(cc)],c,0

def radix_sort(arr):
    if not arr: return [],0,0
    a=arr[:]; c=0; exp=1; mx=max(a)
    while mx//exp>0:
        b=[[] for _ in range(10)]
        for n in a: b[(n//exp)%10].append(n); c+=1
        a=[x for bk in b for x in bk]; exp*=10
    return a,c,0

ALGORITHMS = [
    ("Bubble Sort",       bubble_sort),
    ("Selection Sort",    selection_sort),
    ("Insertion Sort",    insertion_sort),
    ("Merge Sort",        merge_sort),
    ("Quick Sort",        quick_sort),
    ("Random-Quick Sort", random_quick_sort),
    ("Counting Sort",     counting_sort),
    ("Radix Sort",        radix_sort),
]

# ── MST ───────────────────────────────────────────────────────────────
class UF:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        if self.p[x]!=x: self.p[x]=self.find(self.p[x])
        return self.p[x]
    def union(self,x,y):
        rx,ry=self.find(x),self.find(y)
        if rx==ry: return False
        if self.r[rx]<self.r[ry]: rx,ry=ry,rx
        self.p[ry]=rx
        if self.r[rx]==self.r[ry]: self.r[rx]+=1
        return True

def kruskal(verts, edges):
    vi={v:i for i,v in enumerate(verts)}; uf=UF(len(verts)); mst=[]
    for u,v,w in sorted(edges,key=lambda e:e[2]):
        if uf.union(vi[u],vi[v]): mst.append((u,v,w))
        if len(mst)==len(verts)-1: break
    return mst, sum(w for _,_,w in mst)

def prim(verts, edges, start=None):
    adj=defaultdict(list)
    for u,v,w in edges: adj[u].append((v,w)); adj[v].append((u,w))
    s=verts[0] if start is None else start
    vis=set(); mst=[]; h=[(0,None,s)]
    while h:
        w,u,v=heapq.heappop(h)
        if v in vis: continue
        vis.add(v)
        if u is not None: mst.append((u,v,w))
        for nb,wt in adj[v]:
            if nb not in vis: heapq.heappush(h,(wt,v,nb))
    return mst, sum(c for _,_,c in mst)

# ── Recursion profiler ─────────────────────────────────────────────────
class Profiler:
    def __init__(self,fn): self.fn=fn; self.calls=0; self._d=0; self.md=0
    def __call__(self,*a,**kw):
        self.calls+=1; self._d+=1; self.md=max(self.md,self._d)
        r=self.fn(*a,**kw); self._d-=1; return r
    def reset(self): self.calls=self._d=self.md=0

print("✅  Part 4 — Visualization setup complete")
print(f"    seaborn {sns.__version__}  |  matplotlib {matplotlib.__version__}")
print(f"    {len(ALGORITHMS)} sorting algorithms  |  Kruskal + Prim  |  Profiler ready")


---
## Chart 1 — Sorting Benchmark Bars

In [ ]:
# =========================================================
#  CHART 1 — Sorting Benchmark: Horizontal Bar Comparison
#  Shows execution time, comparisons, and swaps side by side.
#  Fastest algorithm highlighted in green.
# =========================================================
N = 1000
random.seed(0)
data = [random.randint(0, 9999) for _ in range(N)]

names_c, times_c, comps_c, swaps_c = [], [], [], []
for name, fn in ALGORITHMS:
    t0 = time.perf_counter()
    _, c, s = fn(data[:])
    names_c.append(name)
    times_c.append((time.perf_counter() - t0) * 1000)
    comps_c.append(c)
    swaps_c.append(s)

short_c = [n.replace(" Sort", "") for n in names_c]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle(f"Chart 1 — Sorting Benchmark  (n = {N})",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

for ax, vals, title, unit, fmt in zip(
        axes,
        [times_c, comps_c, swaps_c],
        ["Execution Time", "Comparisons", "Swaps"],
        ["milliseconds", "count", "count"],
        [".2f", ",d", ",d"]):

    pairs = sorted(zip(vals, short_c), reverse=True)
    v_s, l_s = zip(*pairs)
    fast_idx = len(v_s) - 1
    colors = [PAL_NAMES.get(f"{l} Sort", PALETTE[i % 8]) for i, l in enumerate(l_s)]
    colors = list(colors)
    colors[fast_idx] = GREEN

    bars = ax.barh(l_s, v_s, color=colors, height=0.65,
                   edgecolor="white", linewidth=1.5)
    ax.set_xlim(0, max(v_s) * 1.30)
    ax.set_title(title, color="#111827")
    ax.set_xlabel(unit)
    style_ax(ax, spines=("top", "right", "left"))
    ax.tick_params(left=False)
    label_bars_h(ax, bars, fmt=fmt)

    # Fastest / slowest annotation
    ax.text(v_s[fast_idx] * 0.02, fast_idx,
            "  fastest ✓", va="center", color=GREEN,
            fontsize=8, fontweight="bold", style="italic")
    ax.text(v_s[0] * 0.02, 0,
            "  slowest", va="center", color=RED,
            fontsize=8, fontweight="bold", style="italic")

    # Alternating row tint
    for i, bar in enumerate(bars):
        if i % 2 == 0:
            ax.axhspan(bar.get_y() - 0.02, bar.get_y() + bar.get_height() + 0.02,
                       color="#F8FAFC", zorder=0)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/tmp/p4_chart1.png")
plt.show()
print("Chart 1: benchmark bars ✓")


---
## Chart 2 — Sorting Scalability

In [ ]:
# =========================================================
#  CHART 2 — Sorting Scalability Curve
#  Time vs dataset size for all 8 algorithms.
#  Direct labels on lines; no cluttered legend.
# =========================================================
SIZES = [50, 100, 250, 500, 1000, 2000]
scale = {name: [] for name, _ in ALGORITHMS}
print("Running scalability sweep...")
for sz in SIZES:
    d = [random.randint(0, 9999) for _ in range(sz)]
    for name, fn in ALGORITHMS:
        t0 = time.perf_counter()
        fn(d[:])
        scale[name].append((time.perf_counter() - t0) * 1000)
    print(f"  n = {sz:>5}  ✓")

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor(FIG_BG)

MARKERS = ["o","s","^","D","v","P","X","*"]
for i, (name, _) in enumerate(ALGORITHMS):
    col  = PAL_NAMES.get(name, PALETTE[i])
    yv   = scale[name]
    lw   = 2.8 if name in ("Merge Sort","Quick Sort","Random-Quick Sort",
                            "Counting Sort","Radix Sort") else 1.8
    alpha = 0.98 if lw > 2 else 0.75
    ax.plot(SIZES, yv, marker=MARKERS[i], label=name, color=col,
            linewidth=lw, markersize=7, alpha=alpha,
            markeredgecolor="white", markeredgewidth=1.3)
    # Direct label at end of line
    ax.annotate(
        name.replace(" Sort",""),
        xy=(SIZES[-1], yv[-1]),
        xytext=(6, 0), textcoords="offset points",
        va="center", fontsize=8, color=col, fontweight="600"
    )

# Complexity annotation boxes
ax.text(SIZES[2], ax.get_ylim()[1] * 0.85,
        "O(n²) algorithms", color=RED, fontsize=9, style="italic",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FEF2F2", edgecolor=RED, alpha=0.8))
ax.text(SIZES[4], ax.get_ylim()[1] * 0.12,
        "O(n log n) algorithms", color=GREEN, fontsize=9, style="italic",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#F0FDF4", edgecolor=GREEN, alpha=0.8))

ax.set_title("Chart 2 — Scalability: Execution Time vs Dataset Size",
             color="#111827")
ax.set_xlabel("Dataset Size (n)")
ax.set_ylabel("Time (ms)")
ax.set_xticks(SIZES)
style_ax(ax)
plt.tight_layout()
plt.savefig("/tmp/p4_chart2.png")
plt.show()
print("Chart 2: scalability curve ✓")


---
## Chart 3 — Best / Average / Worst Heatmap

In [ ]:
# =========================================================
#  CHART 3 — Best / Average / Worst Case Heatmap
#  Seaborn heatmap with annotated cells.
#  Darker = slower. Green diagonal = best case per algorithm.
# =========================================================
n_h = 500
cases_h = {
    "Best\n(sorted)":    list(range(n_h)),
    "Average\n(random)": [random.randint(0, 9999) for _ in range(n_h)],
    "Worst\n(reversed)": list(range(n_h, 0, -1)),
}
matrix_h = np.array([
    [(lambda t0, fn=fn, d=d:
      (time.perf_counter() - t0) * 1000)(time.perf_counter(), fn(d[:]))
     for d in cases_h.values()]
    for _, fn in ALGORITHMS
])

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(FIG_BG)

sns.heatmap(
    matrix_h,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="#E2E8F0",
    cbar_kws={"label": "Time (ms)", "shrink": 0.82, "pad": 0.02},
    xticklabels=list(cases_h.keys()),
    yticklabels=[n for n, _ in ALGORITHMS],
    annot_kws={"size": 10, "weight": "bold"},
)
ax.set_title(f"Chart 3 — Best / Average / Worst Case Heatmap  (n = {n_h})",
             color="#111827", pad=14)
ax.set_xlabel("Input Type", labelpad=10)
ax.set_ylabel("Algorithm", labelpad=10)
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

# Add a subtitle
fig.text(0.5, -0.02,
         "Values in ms — darker red = slower. Best case (sorted) shows O(n) for bubble/insertion.",
         ha="center", fontsize=9, color=SLATE, style="italic")
plt.tight_layout()
plt.savefig("/tmp/p4_chart3.png")
plt.show()
print("Chart 3: heatmap (seaborn) ✓")


---
## Chart 4 — MST Graph Visualization

In [ ]:
# =========================================================
#  CHART 4 — MST Graph Visualization
#  NetworkX spring layout with custom node/edge styling.
#  Left: full graph | Centre: Kruskal | Right: Prim
# =========================================================
VERTS = [1,2,3,4,5,6]
EDGES = [(1,2,4),(1,3,3),(2,3,5),(2,4,6),(3,4,7),
         (3,5,8),(4,5,9),(4,6,5),(5,6,6)]

k_mst, k_cost = kruskal(VERTS, EDGES)
p_mst, p_cost = prim(VERTS, EDGES, start=1)

G = nx.Graph(); G.add_nodes_from(VERTS)
for u, v, w in EDGES: G.add_edge(u, v, weight=w)
POS = nx.spring_layout(G, seed=42, k=1.4)

MST_EDGE_COLOR  = "#10B981"
REST_EDGE_COLOR = "#CBD5E1"
NODE_COLOR      = "#1E40AF"
NODE_BORDER     = "white"

def draw_mst_panel(ax, title, subtitle, mst_edges=None):
    mst_set = set()
    if mst_edges:
        for u, v, _ in mst_edges:
            mst_set.add((u, v)); mst_set.add((v, u))

    ecols = [MST_EDGE_COLOR if (u,v) in mst_set else REST_EDGE_COLOR
             for u, v in G.edges()]
    ewids = [4.0 if (u,v) in mst_set else 1.2
             for u, v in G.edges()]
    estyle = ["solid" if (u,v) in mst_set else "dashed"
              for u, v in G.edges()]

    nx.draw_networkx_nodes(G, POS, ax=ax,
                           node_color=NODE_COLOR, node_size=750,
                           linewidths=2.5, edgecolors=NODE_BORDER)
    nx.draw_networkx_labels(G, POS, ax=ax,
                            font_color="white", font_size=12,
                            font_weight="bold")
    for (u, v), col, wid, sty in zip(G.edges(), ecols, ewids, estyle):
        nx.draw_networkx_edges(G, POS, edgelist=[(u,v)], ax=ax,
                               edge_color=col, width=wid,
                               alpha=0.9, style=sty)
    nx.draw_networkx_edge_labels(G, POS,
                                 edge_labels=nx.get_edge_attributes(G,"weight"),
                                 ax=ax, font_size=9, font_color="#374151",
                                 bbox=dict(boxstyle="round,pad=0.2",
                                           facecolor="white",
                                           edgecolor="#E2E8F0",
                                           alpha=0.9))
    ax.set_title(title, color="#111827", pad=10)
    if subtitle:
        ax.set_xlabel(subtitle, color=SLATE, fontsize=9, labelpad=8)
    ax.set_facecolor("white")
    ax.axis("off")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Chart 4 — Minimum Spanning Tree Visualization",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

draw_mst_panel(axes[0], "Full Graph (all edges)", "6 vertices, 9 edges")
draw_mst_panel(axes[1], f"Kruskal's MST",
               f"Total Cost = {k_cost}  |  {len(k_mst)} edges selected", k_mst)
draw_mst_panel(axes[2], "Prim's MST",
               f"Total Cost = {p_cost}  |  {len(p_mst)} edges selected", p_mst)

legend_handles = [
    mpatches.Patch(facecolor=MST_EDGE_COLOR, label="MST edge (selected)"),
    mpatches.Patch(facecolor=REST_EDGE_COLOR, label="Non-MST edge (rejected)"),
]
fig.legend(handles=legend_handles, loc="lower center",
           ncol=2, frameon=True, fontsize=10,
           bbox_to_anchor=(0.5, -0.06),
           edgecolor="#E2E8F0")
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig("/tmp/p4_chart4.png")
plt.show()
print(f"Chart 4: MST visualization ✓  (Kruskal={k_cost}, Prim={p_cost})")


---
## Chart 5 — Recursive Call Count Growth

In [ ]:
# =========================================================
#  CHART 5 — Recursive Call Count Growth
#  Linear + log scale comparison.
#  Danger zone shading shows where naive fibonacci explodes.
# =========================================================
ns = list(range(1, 17))

def _ff(n): return 1 if n<=1 else n*_ff(n-1)
def _bb(n): return n if n<=1 else _bb(n-1)+_bb(n-2)
def _mm(n, c=None):
    if c is None: c={}
    if n in c: return c[n]
    if n<=1: return n
    c[n]=_mm(n-1,c)+_mm(n-2,c); return c[n]

pFF=Profiler(_ff); _ff=pFF
pBB=Profiler(_bb); _bb=pBB
pMM=Profiler(_mm); _mm=pMM

fc5, bc5, mc5 = [], [], []
for n in ns:
    pFF.reset(); _ff(n); fc5.append(pFF.calls)
    pBB.reset(); _bb(n); bc5.append(pBB.calls)
    pMM.reset(); _mm(n); mc5.append(pMM.calls)

SERIES = [
    ("Factorial  O(n)",       fc5, PALETTE[0], "o", "-",  2.5),
    ("Fib Memoized  O(n)",    mc5, PALETTE[2], "s", "--", 2.5),
    ("Fib Naive  O(2^n)",     bc5, RED,        "^", "-",  2.2),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Chart 5 — Recursive Call Count Growth",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

for ax, logy in zip(axes, [False, True]):
    for label, data, col, mk, ls, lw in SERIES:
        ax.plot(ns, data, marker=mk, linestyle=ls, color=col,
                label=label, linewidth=lw, markersize=6, alpha=0.92,
                markeredgecolor="white", markeredgewidth=1.2)

    if logy:
        ax.set_yscale("log")
        ax.set_title("Log Scale — Exponential Gap Visible", color="#111827")
        ax.set_ylabel("Total Calls (log scale)")
    else:
        ax.set_title("Linear Scale", color="#111827")
        ax.set_ylabel("Total Recursive Calls")
        # Shade the danger zone
        thresh_n = next((n for n, c in zip(ns, bc5) if c > 500), None)
        if thresh_n:
            ax.axvspan(thresh_n - 0.5, ns[-1] + 0.5,
                       color="#FEF2F2", alpha=0.4, zorder=0)
            ax.text(thresh_n + 0.2, max(bc5) * 0.75,
                    "Exponential danger zone",
                    color=RED, fontsize=8, style="italic", va="top")

    ax.set_xlabel("n")
    style_ax(ax)
    ax.legend(loc="upper left")
    ax.set_xticks(ns[::2])

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/tmp/p4_chart5.png")
plt.show()
print("Chart 5: recursion call count growth ✓")


---
## Chart 6 — Tower of Hanoi Exponential Growth

In [ ]:
# =========================================================
#  CHART 6 — Tower of Hanoi: Exponential Growth
#  Bar chart + log-scale line to show 2^n - 1 growth clearly.
# =========================================================
def hanoi_count(n):
    m = [0]
    def h(n, s, a, d):
        if n==1: m[0]+=1; return
        h(n-1,s,d,a); m[0]+=1; h(n-1,a,s,d)
    h(n,"A","B","C")
    return m[0]

disks  = list(range(1, 17))
actual = [hanoi_count(d) for d in disks]
theory = [2**d - 1 for d in disks]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Chart 6 — Tower of Hanoi: Move Count = 2^n − 1",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

# Gradient bar chart
rocket_pal = sns.color_palette("rocket_r", len(disks))
bars = axes[0].bar(disks, actual, color=rocket_pal, edgecolor="white",
                   linewidth=1.2, alpha=0.92)
axes[0].plot(disks, theory, "r--o", lw=1.8, ms=5,
             label="Formula: 2^n − 1",
             color="#DC2626", markeredgecolor="white", markeredgewidth=1)
label_bars_v(axes[0], bars, fmt=",d")
axes[0].set_xlabel("Number of Disks")
axes[0].set_ylabel("Moves Required")
axes[0].set_title("Move Count per Disk Count", color="#111827")
axes[0].legend()
style_ax(axes[0])

# Log scale line chart with annotations
axes[1].plot(disks, actual, "o-", color=PALETTE[4], lw=2.5, ms=8,
             label="Actual moves",
             markeredgecolor="white", markeredgewidth=1.5)
axes[1].plot(disks, theory, "--", color=RED, lw=1.8,
             label="2^n − 1 formula")
axes[1].set_yscale("log")
axes[1].set_xlabel("Number of Disks")
axes[1].set_ylabel("Moves (log scale)")
axes[1].set_title("Log Scale — Exponential Curve Shape", color="#111827")
axes[1].legend()
style_ax(axes[1])

# Annotate key points
for d in [4, 8, 12, 16]:
    v = actual[d-1]
    axes[1].annotate(f"n={d}: {v:,}",
                     xy=(d, v), xytext=(8, 4),
                     textcoords="offset points",
                     fontsize=8, color="#374151",
                     arrowprops=dict(arrowstyle="-", color="#9CA3AF", lw=0.8))

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/tmp/p4_chart6.png")
plt.show()
verified = actual == theory
print(f"Chart 6: Hanoi growth ✓  (formula verified: {verified})")


---
## Chart 7 — Recursion Tree Diagrams  *(Bonus)*

In [ ]:
# =========================================================
#  CHART 7 — Recursion Tree Diagrams
#  BONUS: "recursion tree visualization" per spec.
#  Left : Factorial — linear call chain with return values
#  Right: Fibonacci — binary tree showing exponential branching
# =========================================================

fig, axes = plt.subplots(1, 2, figsize=(17, 8))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Chart 7 — Recursion Tree Visualization  (Bonus)",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

# ── Left: Factorial call chain ────────────────────────────────────────
ax = axes[0]
ax.set_facecolor("white")
ax.axis("off")
ax.set_title("Factorial(5) — Linear Call Chain  O(n) depth",
             color="#111827", pad=12)

N_F = 5
BOX_H = 0.75
BOX_W = 8.5
GRAD_COLS = sns.color_palette("Blues_r", N_F + 2)[1:-1]

results_f = {1: 1}
for i in range(2, N_F + 1): results_f[i] = i * results_f[i-1]

for i, val in enumerate(range(N_F, 0, -1)):
    y   = (N_F - i) * (BOX_H + 0.3) + 0.3
    col = list(GRAD_COLS[i])
    rect = plt.Rectangle((0.75, y - BOX_H/2), BOX_W, BOX_H,
                          facecolor=col[:3] + [0.25],
                          edgecolor=col[:3] + [0.9],
                          linewidth=2.0)
    ax.add_patch(rect)
    if val == 1:
        text = f"factorial(1)   →   base case: return 1"
        fw = "bold"
        tc = "#065F46"
    else:
        text = f"factorial({val})   →   {val}  ×  factorial({val-1})  =  {results_f[val]}"
        fw = "normal"
        tc = "#1E3A5F"
    ax.text(0.75 + BOX_W/2, y, text, ha="center", va="center",
            fontsize=10, fontfamily="monospace", fontweight=fw, color=tc)
    # Arrow to next box
    if i < N_F - 1:
        ax.annotate("", xy=(0.75 + BOX_W/2, y - BOX_H/2 - 0.02),
                    xytext=(0.75 + BOX_W/2, y - BOX_H/2 - 0.28),
                    arrowprops=dict(arrowstyle="<-", color="#64748B", lw=1.5))

# Result box
y_res = 0.0
ax.text(0.75 + BOX_W/2, y_res,
        f"Result  =  {results_f[N_F]}",
        ha="center", va="center", fontsize=13,
        fontfamily="monospace", fontweight="bold", color="#065F46",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#DCFCE7",
                  edgecolor="#10B981", linewidth=2.5))

ax.set_xlim(0, 10.5)
ax.set_ylim(-0.8, (N_F + 1) * (BOX_H + 0.3) + 0.3)


# ── Right: Fibonacci binary tree ─────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("white")
ax2.axis("off")
N_B = 5
ax2.set_title(f"Fibonacci({N_B}) — Binary Tree  O(2^n) calls",
              color="#111827", pad=12)

node_data, edge_data = [], []
DEPTH_PAL = sns.color_palette("flare", 6)

def collect(val, depth, x, spread):
    if depth > 4: return
    node_data.append((x, -depth * 1.5, val, depth))
    if val > 1:
        lx, rx = x - spread, x + spread
        edge_data.append(((x, -depth*1.5), (lx, -(depth+1)*1.5)))
        edge_data.append(((x, -depth*1.5), (rx, -(depth+1)*1.5)))
        collect(val-1, depth+1, lx, spread*0.48)
        collect(val-2, depth+1, rx, spread*0.48)

collect(N_B, 0, 0, 4.5)

for (x1,y1),(x2,y2) in edge_data:
    ax2.plot([x1,x2],[y1,y2], color="#CBD5E1", lw=1.4, zorder=1, alpha=0.8)

for x, y, val, depth in node_data:
    is_base = val <= 1
    face_col = "#DCFCE7" if is_base else list(DEPTH_PAL[depth % len(DEPTH_PAL)])[:3]
    edge_col = "#10B981" if is_base else list(DEPTH_PAL[depth % len(DEPTH_PAL)])[:3]
    radius   = 0.38 if depth == 0 else 0.30
    circle   = plt.Circle((x, y), radius,
                           facecolor=face_col, edgecolor=edge_col,
                           linewidth=2.0, zorder=2)
    ax2.add_patch(circle)
    ax2.text(x, y, f"f({val})", ha="center", va="center",
             fontsize=8 if depth > 0 else 10,
             fontfamily="monospace", fontweight="bold",
             color="#065F46" if is_base else "#1E3A5F", zorder=3)

legend2 = [
    mpatches.Patch(facecolor="#DCFCE7", edgecolor="#10B981",
                   label="Base case  f(0) or f(1)"),
    mpatches.Patch(facecolor=list(DEPTH_PAL[0])[:3] + [0.4],
                   edgecolor=list(DEPTH_PAL[0])[:3],
                   label="Recursive calls"),
]
ax2.legend(handles=legend2, loc="lower center",
           bbox_to_anchor=(0.5, -0.06), ncol=2, fontsize=9,
           framealpha=0.92, edgecolor="#E2E8F0")
ax2.set_xlim(-9, 9)
ax2.set_ylim(-N_B*1.5 - 1.5, 1.2)

plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.savefig("/tmp/p4_chart7.png")
plt.show()
print("Chart 7: recursion tree diagrams ✓")
print(f"  Factorial({N_F}): linear chain, depth = {N_F}")
print(f"  Fibonacci({N_B}): binary branching tree (tree shown to depth 4)")


---
## Chart 8 — Sorting Metrics Full Breakdown

In [ ]:
# =========================================================
#  CHART 8 — Sorting Metrics: Full Breakdown
#  Grouped bar chart: time + comps + swaps per algorithm.
#  Separate subplots with consistent color coding.
# =========================================================
random.seed(0)
data8 = [random.randint(0, 9999) for _ in range(1000)]

names8, times8, comps8, swaps8 = [], [], [], []
for name, fn in ALGORITHMS:
    t0 = time.perf_counter()
    _, c, s = fn(data8[:])
    names8.append(name.replace(" Sort", ""))
    times8.append((time.perf_counter() - t0) * 1000)
    comps8.append(c)
    swaps8.append(s)

x8 = np.arange(len(names8))
COLS8 = [PAL_NAMES.get(f"{n} Sort", PALETTE[i]) for i, n in enumerate(names8)]

fig, axes = plt.subplots(3, 1, figsize=(13, 13))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Chart 8 — Sorting Metrics Breakdown  (n = 1000)",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

for ax, vals, title, unit, fmt in zip(
        axes,
        [times8, comps8, swaps8],
        ["Execution Time (ms)", "Number of Comparisons", "Number of Swaps"],
        ["ms", "count", "count"],
        [".2f", ",d", ",d"]):

    bars = ax.bar(x8, vals, color=COLS8, edgecolor="white",
                  linewidth=1.5, alpha=0.92, width=0.68)
    ax.set_title(title, color="#111827")
    ax.set_ylabel(unit)
    ax.set_xticks(x8)
    ax.set_xticklabels(names8, rotation=25, ha="right", fontsize=9)
    style_ax(ax)
    label_bars_v(ax, bars, fmt=fmt)

    # Zero-swap annotation
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v == 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    ax.get_ylim()[1] * 0.04,
                    "zero", ha="center", va="bottom",
                    fontsize=8, color=GREEN, fontweight="bold")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/tmp/p4_chart8.png")
plt.show()
print("Chart 8: sorting metrics breakdown ✓")

# Summary observations
print()
print("  Observations:")
max_t = names8[times8.index(max(times8))]; min_t = names8[times8.index(min(times8))]
max_c = names8[comps8.index(max(comps8))]; min_c = names8[comps8.index(min(c for c in comps8 if c>0))]
print(f"  Slowest: {max_t}  ({max(times8):.2f} ms)")
print(f"  Fastest: {min_t}  ({min(times8):.2f} ms)")
print(f"  Most comparisons:  {max_c}  ({max(comps8):,})")
print(f"  Fewest comparisons: {min_c}  ({min(c for c in comps8 if c>0):,})")
zero_s = [n for n, s in zip(names8, swaps8) if s == 0]
print(f"  Zero swaps (non-comparison): {', '.join(zero_s)}")


---
## Chart 9 — MST Cost Comparison

In [ ]:
# =========================================================
#  CHART 9 — MST Cost: Kruskal vs Prim on Random Graphs
#  Confirms both algorithms always find the same minimum cost.
#  Left: cost vs graph size line | Right: grouped bar chart
# =========================================================
np.random.seed(7)

def random_graph(n_v, extra=5):
    verts = list(range(1, n_v + 1))
    edict = {}
    for i in range(1, n_v):
        edict[(i, i+1)] = int(np.random.randint(1, 21))
    attempts = 0
    while len(edict) < n_v - 1 + extra and attempts < 500:
        attempts += 1
        u = int(np.random.randint(1, n_v+1))
        v = int(np.random.randint(1, n_v+1))
        if u != v:
            k = (min(u,v), max(u,v))
            if k not in edict:
                edict[k] = int(np.random.randint(1, 21))
    return verts, [(u,v,w) for (u,v),w in edict.items()]

G_SIZES = [4, 5, 6, 7, 8, 9, 10, 12, 14, 16]
k_costs, p_costs, agree_all = [], [], True

print(f"  {'Vertices':>10}  {'Edges':>6}  {'Kruskal':>9}  {'Prim':>7}  Agree?")
print("  " + "-" * 44)
for n_v in G_SIZES:
    verts, edges = random_graph(n_v, extra=n_v)
    _, kc = kruskal(verts, edges)
    _, pc = prim(verts, edges)
    ok = kc == pc
    if not ok: agree_all = False
    k_costs.append(kc); p_costs.append(pc)
    print(f"  {n_v:>10}  {len(edges):>6}  {kc:>9}  {pc:>7}  {'✓ YES' if ok else '✗ NO'}")

print()
print(f"  All costs agree: {'✓ YES — both algorithms always find the optimal MST' if agree_all else '✗ NO'}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Chart 9 — MST Cost: Kruskal's vs Prim's  (10 random graphs)",
             fontsize=15, fontweight="bold", color="#111827", y=1.01)

# Line chart
kc_col = PAL_NAMES["Kruskal"]
pc_col = PAL_NAMES["Prim"]
axes[0].plot(G_SIZES, k_costs, "o-", color=kc_col, lw=2.5, ms=8,
             label="Kruskal's", markeredgecolor="white", markeredgewidth=1.5)
axes[0].plot(G_SIZES, p_costs, "s--", color=pc_col, lw=2.5, ms=8,
             label="Prim's", markeredgecolor="white", markeredgewidth=1.5)
axes[0].fill_between(G_SIZES, k_costs, p_costs, alpha=0.08, color="#6366F1")
axes[0].set_xlabel("Number of Vertices")
axes[0].set_ylabel("MST Total Cost")
axes[0].set_title("MST Cost vs Graph Size", color="#111827")
axes[0].legend()
style_ax(axes[0])
axes[0].text(G_SIZES[3], max(k_costs) * 0.92,
             "Lines overlap = same optimal cost",
             fontsize=8.5, color=SLATE, style="italic",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                       edgecolor="#E2E8F0", alpha=0.9))

# Grouped bar chart
xg = np.arange(len(G_SIZES))
axes[1].bar(xg - 0.22, k_costs, width=0.40, color=kc_col,
            label="Kruskal's", alpha=0.90, edgecolor="white", linewidth=1.2)
axes[1].bar(xg + 0.22, p_costs, width=0.40, color=pc_col,
            label="Prim's",    alpha=0.90, edgecolor="white", linewidth=1.2)
axes[1].set_xticks(xg)
axes[1].set_xticklabels([str(s) for s in G_SIZES])
axes[1].set_xlabel("Vertices")
axes[1].set_ylabel("MST Total Cost")
axes[1].set_title("Side-by-Side per Graph", color="#111827")
axes[1].legend()
style_ax(axes[1])

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/tmp/p4_chart9.png")
plt.show()
print("Chart 9: MST cost comparison ✓")
